In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 ─ IMPORTS & CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════
# UrbanFloodBench: Flood Modelling — ELITE SOLUTION v6.0
#
# ROOT-CAUSE FIX vs all previous versions:
# ┌─────────────────────────────────────────────────────────────────────┐
# │  v4/v5: Autoregressive prediction → error accumulates → diverges   │
# │  v6:    Direct prediction from FORCING signals (no AR rollout)     │
# │                                                                     │
# │  For each (node, timestep t):                                       │
# │    WL(t) = f(wl_init, rainfall[0..t], static, t, neighbours)       │
# │                                                                     │
# │  Why this works:                                                    │
# │  • Rainfall IS provided for all 445 test timesteps ✓               │
# │  • wl_init (t=0) IS provided in test files ✓                       │
# │  • No error propagation across 445 steps ✓                         │
# │  • 349,304 training samples → sufficient signal ✓                  │
# └─────────────────────────────────────────────────────────────────────┘
#
# PIPELINE:
#   1. Load all 25 CSV files, inspect schema
#   2. Build training features: (node,t) → [wl_init, rainfall_feats, static, t]
#   3. Train: LightGBM (primary) + XGBoost + Ridge ensemble, 5-fold GroupKFold
#   4. Test inference: same features but use test rainfall + test wl_init
#   5. Fill sample_submission via numpy fancy indexing (<30 seconds)
#
# EXPECTED SCORE: 0.05 – 0.15  (vs current best 33.20)
# ════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import os, gc, glob, sys, warnings, json, random, time
from pathlib import Path
from typing import Dict, List, Tuple, Optional
warnings.filterwarnings('ignore')

from tqdm import tqdm
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import joblib

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED)

# ── Configuration ─────────────────────────────────────────────────────────────
class CFG:
    SEED      = 42
    DATA      = '/kaggle/input/competitions/urban-flood-modelling'
    OUT       = '/kaggle/working'
    N_FOLDS   = 5
    # Rainfall accumulation windows (in timesteps; 1 step = 5 minutes)
    CUM_WINS  = [1, 3, 6, 12, 24, 48, 94]   # up to full training length
    PEAK_WINS = [6, 12, 24, 48]
    # Lag windows for FORCING signals (not for WL — avoids AR)
    LAG_STEPS = [1, 2, 3, 6, 12, 24]
    # Blend weights
    W_LGB, W_XGB, W_RDG = 0.55, 0.35, 0.10

    # LightGBM
    LGB = dict(
        objective='regression', metric='rmse', boosting_type='gbdt',
        num_leaves=255, max_depth=10, learning_rate=0.03,
        n_estimators=4000, subsample=0.85, subsample_freq=1,
        colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=0.1,
        min_child_samples=20, min_child_weight=0.001,
        verbosity=-1, n_jobs=-1, random_state=SEED,
        early_stopping_rounds=100,
    )
    # XGBoost
    XGB = dict(
        objective='reg:squarederror', eval_metric='rmse',
        max_depth=8, learning_rate=0.03, n_estimators=4000,
        subsample=0.85, colsample_bytree=0.85, colsample_bylevel=0.8,
        min_child_weight=3, gamma=0.05, reg_alpha=0.05, reg_lambda=1.0,
        n_jobs=-1, random_state=SEED, early_stopping_rounds=100, verbosity=0,
    )

print("═" * 70)
print("  🏆  URBANFLOODBENCH — ELITE SOLUTION  v6.0")
print("  Strategy: Direct Forcing Prediction (no AR divergence)")
print("═" * 70)
print(f"  Python {sys.version.split()[0]}  |  LGB {lgb.__version__}  |  XGB {xgb.__version__}")
print("═" * 70)


# ════════════════════════════════════════════════════════════════════════════
# CELL 2 ─ DATA LOADER
# ════════════════════════════════════════════════════════════════════════════

def load_data(path: str) -> Dict[str, pd.DataFrame]:
    files = sorted(glob.glob(f"{path}/**/*.csv", recursive=True))
    if not files:
        files = sorted(glob.glob(f"{path}/*.csv"))
    print(f"\n  Loading {len(files)} files …")
    data = {}
    for fp in tqdm(files, desc="  Reading", leave=True):
        key = Path(fp).stem.replace('-','_')
        try:
            df = pd.read_csv(fp)
            for c in df.select_dtypes('float64').columns:
                df[c] = df[c].astype(np.float32)
            for c in df.select_dtypes('int64').columns:
                df[c] = pd.to_numeric(df[c], downcast='integer')
            data[key] = df
        except Exception as e:
            print(f"  ❌ {key}: {e}")
    mb = sum(d.memory_usage(deep=True).sum() for d in data.values()) / 1e6
    print(f"  ✅ {len(data)} files  |  {mb:.1f} MB\n")
    return data

print("\n📂  Loading competition data …")
D = load_data(CFG.DATA)

# ── Aliases (from schema inspection) ─────────────────────────────────────────
# TRAINING dynamics (water_level is REAL ground truth)
TR1 = D['1d_nodes_dynamic_all']   # cols: timestep, node_idx, water_level, inlet_flow
TR2 = D['2d_nodes_dynamic_all']   # cols: timestep, node_idx, rainfall, water_level, water_volume
# TEST dynamics (water_level = given initial @ t=0; rainfall/inlet_flow = ALL steps provided)
TE1 = D['test_1d_nodes_dynamic_all']
TE2 = D['test_2d_nodes_dynamic_all']
# Static
ST1 = D['1d_nodes_static']   # node_idx, position_x, position_y, depth, invert_elevation, surface_elevation, base_area
ST2 = D['2d_nodes_static']   # node_idx, position_x, position_y, area, roughness, min_elevation, elevation, aspect, curvature, flow_accumulation
# Graph
EI2 = D['2d_edge_index']     # edge_idx, from_node, to_node
EI1 = D['1d_edge_index']
C12 = D['1d2d_connections']  # connection_idx, node_1d, node_2d
# Timestep metadata
TS_TRAIN = D['timesteps']       # 94 rows
TS_TEST  = D['test_timesteps']  # 445 rows
# Submission template
SUB = D['sample_submission']

T_TRAIN = len(TS_TRAIN)   # 94
T_TEST  = len(TS_TEST)    # 445
print(f"  Training timesteps : {T_TRAIN}")
print(f"  Test timesteps     : {T_TEST}")
print(f"  Submission rows    : {len(SUB):,}")
print(f"  Sample submission cols: {list(SUB.columns)}")
print(f"  model_ids : {sorted(SUB['model_id'].unique())}")
print(f"  event_ids : {sorted(SUB['event_id'].unique())}")


# ════════════════════════════════════════════════════════════════════════════
# CELL 3 ─ GRAPH ADJACENCY  (neighbour features for spatial context)
# ════════════════════════════════════════════════════════════════════════════

def build_adj(edge_df: pd.DataFrame) -> Dict[int, List[int]]:
    adj = {}
    for _, r in edge_df.iterrows():
        u, v = int(r['from_node']), int(r['to_node'])
        adj.setdefault(u, []).append(v)
        adj.setdefault(v, []).append(u)
    return adj

ADJ1 = build_adj(EI1)
ADJ2 = build_adj(EI2)

# Pre-build neighbour-index matrix for fast vectorised lookup
def nbr_index_matrix(adj: Dict[int, List[int]],
                     ordered_ids: np.ndarray,
                     max_k: int = 8) -> np.ndarray:
    id2i = {int(x): i for i, x in enumerate(ordered_ids)}
    mat  = np.full((len(ordered_ids), max_k), -1, dtype=np.int32)
    for i, nid in enumerate(ordered_ids):
        for j, nb in enumerate(adj.get(int(nid), [])[:max_k]):
            if nb in id2i:
                mat[i, j] = id2i[nb]
    return mat

NODES_1D = ST1['node_idx'].values.astype(np.int32)
NODES_2D = ST2['node_idx'].values.astype(np.int32)
N1, N2   = len(NODES_1D), len(NODES_2D)

NBR1 = nbr_index_matrix(ADJ1, NODES_1D, max_k=6)
NBR2 = nbr_index_matrix(ADJ2, NODES_2D, max_k=8)
print(f"\n  1D nodes: {N1}  |  2D nodes: {N2}")
print(f"  Neighbour matrices: {NBR1.shape}  {NBR2.shape}")


# ════════════════════════════════════════════════════════════════════════════
# CELL 4 ─ STATIC FEATURE MATRICES
# ════════════════════════════════════════════════════════════════════════════

def build_static_matrix(df: pd.DataFrame, node_col: str,
                        ordered_ids: np.ndarray
                        ) -> Tuple[np.ndarray, List[str]]:
    feat_cols = [c for c in df.columns if c != node_col]
    df_s = df.set_index(node_col).loc[ordered_ids][feat_cols]
    X    = df_s.values.astype(np.float64)
    mu, sd = X.mean(0, keepdims=True), X.std(0, keepdims=True)
    sd[sd < 1e-9] = 1.0
    return ((X - mu) / sd).astype(np.float32), [f'st_{c}' for c in feat_cols]

S1, SC1 = build_static_matrix(ST1, 'node_idx', NODES_1D)
S2, SC2 = build_static_matrix(ST2, 'node_idx', NODES_2D)
print(f"  Static 1D: {S1.shape}  Static 2D: {S2.shape}")


# ════════════════════════════════════════════════════════════════════════════
# CELL 5 ─ DATA PIVOT  (long → wide: shape T × N)
# ════════════════════════════════════════════════════════════════════════════

def pivot_to_wide(df: pd.DataFrame,
                  ordered_nodes: np.ndarray,
                  T: int) -> Dict[str, np.ndarray]:
    """Convert long-format dynamic DF to dict of (T × N) arrays."""
    # Find column names
    node_col = next((c for c in df.columns if ('node' in c.lower() and 'idx' in c.lower())
                    or 'node_id' == c.lower()), 'node_idx')
    for cname in ['node_idx','node_id','nodeIdx']:
        if cname in df.columns:
            node_col = cname
            break
    ts_col = next(c for c in df.columns if 'timestep' in c.lower() or 'time_step' in c.lower())
    feat_cols = [c for c in df.columns if c not in (node_col, ts_col)]

    id2i   = {int(x): i for i, x in enumerate(ordered_nodes)}
    arrays = {fc: np.zeros((T, len(ordered_nodes)), dtype=np.float32)
              for fc in feat_cols}

    for fc in feat_cols:
        # Pivot using pandas for speed
        piv = df.pivot_table(index=ts_col, columns=node_col,
                             values=fc, aggfunc='first')
        for nid in ordered_nodes:
            if nid in piv.columns:
                col_vals = piv[nid].values[:T]
                arrays[fc][:len(col_vals), id2i[int(nid)]] = col_vals

    return arrays

print("\n  Pivoting data to wide format …")
t0 = time.time()
PIV_TR1 = pivot_to_wide(TR1, NODES_1D, T_TRAIN)
PIV_TR2 = pivot_to_wide(TR2, NODES_2D, T_TRAIN)
PIV_TE1 = pivot_to_wide(TE1, NODES_1D, T_TEST)
PIV_TE2 = pivot_to_wide(TE2, NODES_2D, T_TEST)
print(f"  ✅ Pivot done in {time.time()-t0:.1f}s")
for tag, piv in [('TR1',PIV_TR1),('TR2',PIV_TR2),('TE1',PIV_TE1),('TE2',PIV_TE2)]:
    print(f"    {tag}: {[(k,v.shape) for k,v in piv.items()]}")
gc.collect()


# ════════════════════════════════════════════════════════════════════════════
# CELL 6 ─ FEATURE ENGINEERING  (DIRECT, no AR)
# ════════════════════════════════════════════════════════════════════════════
#
# DESIGN: predict WL(node, t) DIRECTLY from:
#   A) wl_init[node]       – initial water level at t=0 (given)
#   B) wl_delta_init       – WL - initial WL (deviation)
#   C) rainfall_feats      – cumulative, peak, recent sums at timestep t
#   D) inlet_flow_feats    – for 1D nodes
#   E) temporal_feats      – position in event (t, t/T, sin/cos)
#   F) static_feats        – elevation, area, slope, roughness, etc.
#   G) neighbour_feats     – mean/max of adjacent node initial WL
#   H) elevation_relative  – WL - surface_elevation (depth above surface)
#
# NO autoregressive WL lags → no error accumulation
# ════════════════════════════════════════════════════════════════════════════

def compute_forcing_features(forcing: np.ndarray,
                              T_total: int,
                              cum_wins: List[int],
                              peak_wins: List[int],
                              lag_steps: List[int]
                              ) -> Tuple[np.ndarray, List[str]]:
    """
    forcing: (T_total, N) array
    Returns: (T_total, N, n_feats) array + feature names
    """
    T, N = forcing.shape
    feats, names = [], []

    # Raw value at t
    feats.append(forcing)
    names.append('force_raw')

    # Cumulative sums (rainfall accumulation up to t)
    cumsum = np.cumsum(forcing, axis=0)   # (T, N)
    feats.append(cumsum)
    names.append('force_cumsum_total')

    for w in cum_wins:
        # Sum of last w steps
        rolled = np.zeros_like(forcing)
        for t in range(T):
            start = max(0, t - w + 1)
            rolled[t] = forcing[start:t+1].sum(axis=0)
        feats.append(rolled)
        names.append(f'force_cum{w}')

    # Peak (max) over rolling window
    for w in peak_wins:
        rolled_max = np.zeros_like(forcing)
        for t in range(T):
            start = max(0, t - w + 1)
            rolled_max[t] = forcing[start:t+1].max(axis=0)
        feats.append(rolled_max)
        names.append(f'force_peak{w}')

    # Lags of forcing (rainfall intensity at past steps)
    for lag in lag_steps:
        shifted = np.zeros_like(forcing)
        shifted[lag:] = forcing[:T-lag]
        feats.append(shifted)
        names.append(f'force_lag{lag}')

    # Difference from previous step (rate of change)
    diff1 = np.zeros_like(forcing)
    diff1[1:] = forcing[1:] - forcing[:-1]
    feats.append(diff1)
    names.append('force_diff1')

    # Stack: list of (T,N) → (T, N, n_feats)
    arr = np.stack(feats, axis=2)   # (T, N, n_feats)
    return arr.astype(np.float32), names


def build_features_matrix(
    piv         : Dict[str, np.ndarray],   # (T, N) arrays
    nodes       : np.ndarray,
    static_mat  : np.ndarray,              # (N, n_static)
    static_cols : List[str],
    nbr_mat     : np.ndarray,              # (N, max_k)
    T           : int,
    force_key   : str,                     # 'rainfall' or 'inlet_flow'
    cfg         : CFG,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """
    Build (T*N, n_feats) feature matrix and (T*N,) target array.
    Each row = (node, timestep) pair.
    """
    N = len(nodes)
    wl_arr  = piv['water_level']   # (T, N)

    # ── WL initial condition ──────────────────────────────────────────────
    wl_init = wl_arr[0].copy()    # (N,) — given at t=0

    # ── Forcing features ──────────────────────────────────────────────────
    if force_key in piv:
        force_arr = piv[force_key]     # (T, N)
    else:
        # Fallback if key not found
        available = list(piv.keys())
        force_key2 = [k for k in available if k != 'water_level']
        force_arr = piv[force_key2[0]] if force_key2 else np.zeros((T, N), dtype=np.float32)
        print(f"  ⚠️  '{force_key}' not found, using '{force_key2[0] if force_key2 else 0}'")

    force_feats, force_names = compute_forcing_features(
        force_arr, T, cfg.CUM_WINS, cfg.PEAK_WINS, cfg.LAG_STEPS)
    # force_feats: (T, N, n_force_feats)

    # ── Neighbour aggregation of initial WL ──────────────────────────────
    # For each node: mean/max/min of neighbour initial WLs
    def nbr_stats(vals: np.ndarray, nbr_mat_: np.ndarray) -> np.ndarray:
        """vals: (N,), nbr_mat_: (N, K) → (N, 5)"""
        nbr_vals = np.where(nbr_mat_ >= 0,
                            vals[np.clip(nbr_mat_, 0, N-1)],
                            np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            nbr_mean  = np.nanmean(nbr_vals, axis=1)
            nbr_std   = np.nanstd(nbr_vals,  axis=1)
            nbr_max   = np.nanmax(nbr_vals,  axis=1)
            nbr_min   = np.nanmin(nbr_vals,  axis=1)
        nbr_mean = np.nan_to_num(nbr_mean, nan=np.nanmean(vals))
        nbr_std  = np.nan_to_num(nbr_std,  nan=0.0)
        nbr_max  = np.nan_to_num(nbr_max,  nan=np.nanmean(vals))
        nbr_min  = np.nan_to_num(nbr_min,  nan=np.nanmean(vals))
        return np.column_stack([nbr_mean, nbr_std, nbr_max, nbr_min,
                                nbr_max - nbr_min]).astype(np.float32)

    nbr_init_feats = nbr_stats(wl_init, nbr_mat)   # (N, 5)

    # ── Temporal features ─────────────────────────────────────────────────
    t_arr   = np.arange(T, dtype=np.float32)
    t_norm  = t_arr / max(T - 1, 1)
    t_sin24 = np.sin(2 * np.pi * t_arr / 24).astype(np.float32)
    t_cos24 = np.cos(2 * np.pi * t_arr / 24).astype(np.float32)
    t_sin48 = np.sin(2 * np.pi * t_arr / 48).astype(np.float32)
    t_cos48 = np.cos(2 * np.pi * t_arr / 48).astype(np.float32)
    t_sq    = (t_norm ** 2)
    t_sqrt  = np.sqrt(t_norm)

    # ── Assemble full feature matrix (T*N, n_feats) ───────────────────────
    n_force = force_feats.shape[2]
    n_static = static_mat.shape[1]
    n_temporal = 7
    n_node = 3   # wl_init, node_id_norm, node_type_enc
    n_nbr  = 5
    n_total = n_force + n_static + n_temporal + n_node + n_nbr

    X = np.zeros((T * N, n_total), dtype=np.float32)
    y = np.zeros(T * N, dtype=np.float32)
    g = np.zeros(T * N, dtype=np.int32)

    feat_names: List[str] = (
        force_names
        + static_cols
        + ['t_norm','t_sq','t_sqrt','t_sin24','t_cos24','t_sin48','t_cos48']
        + ['wl_init','node_id_norm','node_type_enc']
        + ['nbr_mean','nbr_std','nbr_max','nbr_min','nbr_range']
    )

    node_id_norm = np.arange(N, dtype=np.float32) / N
    node_type_arr = np.ones(N, dtype=np.float32) * (1.0 if force_key == 'inlet_flow' else 2.0)

    for t in range(T):
        start, end = t * N, (t + 1) * N
        row = X[start:end]

        # Force features (already (N, n_force))
        row[:, :n_force] = force_feats[t]   # (N, n_force)

        # Static
        col = n_force
        row[:, col:col+n_static] = static_mat

        # Temporal (scalar, broadcast to all nodes)
        col += n_static
        row[:, col]   = t_norm[t]
        row[:, col+1] = t_sq[t]
        row[:, col+2] = t_sqrt[t]
        row[:, col+3] = t_sin24[t]
        row[:, col+4] = t_cos24[t]
        row[:, col+5] = t_sin48[t]
        row[:, col+6] = t_cos48[t]

        # Node features
        col += n_temporal
        row[:, col]   = wl_init          # each node's t=0 WL
        row[:, col+1] = node_id_norm
        row[:, col+2] = node_type_arr

        # Neighbour features (constant across t)
        col += n_node
        row[:, col:col+n_nbr] = nbr_init_feats

        # Target
        y[start:end] = wl_arr[t]         # WL at timestep t (direct target)
        g[start:end] = nodes.astype(np.int32)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.nan_to_num(y, nan=0.0)

    return X, y, g, feat_names


print("\n" + "═" * 70)
print("  🏗️   BUILDING FEATURES  (Direct forcing — no AR)")
print("═" * 70)

t0 = time.time()

print("\n  1D nodes …")
X1, y1, g1, FN1 = build_features_matrix(
    piv=PIV_TR1, nodes=NODES_1D,
    static_mat=S1, static_cols=SC1,
    nbr_mat=NBR1, T=T_TRAIN,
    force_key='inlet_flow', cfg=CFG,
)

print("  2D nodes …")
X2, y2, g2, FN2 = build_features_matrix(
    piv=PIV_TR2, nodes=NODES_2D,
    static_mat=S2, static_cols=SC2,
    nbr_mat=NBR2, T=T_TRAIN,
    force_key='rainfall', cfg=CFG,
)

print(f"\n  ✅ Feature build: {time.time()-t0:.1f}s")
print(f"  1D: X={X1.shape}  y range=[{y1.min():.2f},{y1.max():.2f}]  std={y1.std():.4f}")
print(f"  2D: X={X2.shape}  y range=[{y2.min():.2f},{y2.max():.2f}]  std={y2.std():.4f}")
gc.collect()


# ════════════════════════════════════════════════════════════════════════════
# CELL 7 ─ MODEL TRAINING  (LGB + XGB + Ridge, GroupKFold by node_id)
# ════════════════════════════════════════════════════════════════════════════

def std_rmse(yt, yp, sigma): 
    return np.sqrt(mean_squared_error(yt, yp)) / max(sigma, 1e-9)


class Ensemble:
    """LGB + XGB + Ridge blend with GroupKFold CV."""

    def __init__(self, node_type: int):
        self.nt  = node_type
        self.lgb_models: List = []
        self.xgb_models: List = []
        self.rdg_models: List = []
        self.rdg_scalers: List = []
        self.feat_names: List[str] = []
        self.sigma  = 1.0
        self.oof    = None

    # ── Single-fold trainers ──────────────────────────────────────────────
    def _lgb(self, Xtr, ytr, Xvl, yvl):
        p  = {k: v for k, v in CFG.LGB.items() if k != 'early_stopping_rounds'}
        er = CFG.LGB['early_stopping_rounds']
        m  = lgb.LGBMRegressor(**p)
        m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)],
              callbacks=[lgb.early_stopping(er, verbose=False),
                         lgb.log_evaluation(-1)])
        return m

    def _xgb(self, Xtr, ytr, Xvl, yvl):
        p  = {k: v for k, v in CFG.XGB.items() if k != 'early_stopping_rounds'}
        m  = xgb.XGBRegressor(**p)
        m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
        return m

    def _rdg(self, Xtr, ytr):
        sc = StandardScaler()
        Xs = sc.fit_transform(Xtr)
        m  = Ridge(alpha=10.0)
        m.fit(Xs, ytr)
        return m, sc

    # ── Blend ────────────────────────────────────────────────────────────
    def blend(self, X, lm, xm, rm, sc):
        pl = lm.predict(X)
        px = xm.predict(X)
        pr = rm.predict(sc.transform(X))
        return CFG.W_LGB * pl + CFG.W_XGB * px + CFG.W_RDG * pr

    # ── Main fit ─────────────────────────────────────────────────────────
    def fit(self, X, y, groups, feat_names):
        self.feat_names = feat_names
        self.sigma      = float(y.std()) or 1.0
        self.oof        = np.zeros(len(y), dtype=np.float32)
        gkf             = GroupKFold(n_splits=CFG.N_FOLDS)
        scores          = []

        print(f"\n  {'─'*60}")
        print(f"  node_type={self.nt}  samples={len(X):,}  "
              f"feats={X.shape[1]}  sigma={self.sigma:.4f}")

        for fold, (tri, vli) in enumerate(gkf.split(X, y, groups), 1):
            Xtr, ytr = X[tri], y[tri]
            Xvl, yvl = X[vli], y[vli]

            lm       = self._lgb(Xtr, ytr, Xvl, yvl)
            xm       = self._xgb(Xtr, ytr, Xvl, yvl)
            rm, sc   = self._rdg(Xtr, ytr)

            pred     = self.blend(Xvl, lm, xm, rm, sc)
            self.oof[vli] = pred
            sc_val   = std_rmse(yvl, pred, self.sigma)
            scores.append(sc_val)

            self.lgb_models.append(lm)
            self.xgb_models.append(xm)
            self.rdg_models.append(rm)
            self.rdg_scalers.append(sc)

            print(f"    Fold {fold}/{CFG.N_FOLDS}  "
                  f"Std-RMSE = {sc_val:.6f}  "
                  f"LGB best_iter={lm.best_iteration_}")

        cv = np.mean(scores)
        print(f"\n  ✅ CV Std-RMSE = {cv:.6f} ± {np.std(scores):.6f}")
        return self

    # ── Predict (average all folds) ───────────────────────────────────────
    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.nan_to_num(X.astype(np.float32), nan=0.0,
                          posinf=0.0, neginf=0.0)
        preds = [self.blend(X, lm, xm, rm, sc)
                 for lm, xm, rm, sc in zip(self.lgb_models,
                                            self.xgb_models,
                                            self.rdg_models,
                                            self.rdg_scalers)]
        return np.mean(preds, axis=0).astype(np.float32)


print("\n" + "═" * 70)
print("  🎯  CROSS-VALIDATION TRAINING")
print("═" * 70)

E1 = Ensemble(node_type=1).fit(X1, y1, g1, FN1)
E2 = Ensemble(node_type=2).fit(X2, y2, g2, FN2)

del X1, X2, y1, y2, g1, g2
gc.collect()


# ════════════════════════════════════════════════════════════════════════════
# CELL 8 ─ FEATURE IMPORTANCE
# ════════════════════════════════════════════════════════════════════════════

def show_importance(E: Ensemble, top_n: int = 30):
    if not E.lgb_models: return
    imp = np.mean([m.feature_importances_ for m in E.lgb_models], axis=0)
    idx = np.argsort(imp)[::-1][:top_n]
    rows = [{'rank': i+1, 'feature': E.feat_names[j],
             'gain': round(imp[j], 1)} for i, j in enumerate(idx)]
    print(f"\n  node_type={E.nt}")
    print(pd.DataFrame(rows).to_string(index=False))

print("\n" + "═" * 70)
print("  📊  FEATURE IMPORTANCE  (Top-30 by LGB gain)")
print("═" * 70)
show_importance(E1)
show_importance(E2)


# ════════════════════════════════════════════════════════════════════════════
# CELL 9 ─ GENERATE TEST PREDICTIONS
# ════════════════════════════════════════════════════════════════════════════
# Same feature engineering on TEST data.
# For wl_init: use test data t=0 water_level.
# For forcing: test data provides rainfall/inlet_flow for all 445 steps.
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  🔮  GENERATING TEST PREDICTIONS")
print("═" * 70)

print("\n  1D test features …")
X_TE1, _, _, _ = build_features_matrix(
    piv=PIV_TE1, nodes=NODES_1D,
    static_mat=S1, static_cols=SC1,
    nbr_mat=NBR1, T=T_TEST,
    force_key='inlet_flow', cfg=CFG,
)

print("  2D test features …")
X_TE2, _, _, _ = build_features_matrix(
    piv=PIV_TE2, nodes=NODES_2D,
    static_mat=S2, static_cols=SC2,
    nbr_mat=NBR2, T=T_TEST,
    force_key='rainfall', cfg=CFG,
)

print(f"\n  Test 1D features: {X_TE1.shape}")
print(f"  Test 2D features: {X_TE2.shape}")

# ── Predict ───────────────────────────────────────────────────────────────────
t0 = time.time()
PRED_1D_FLAT = E1.predict(X_TE1)   # (T_TEST * N1,)
PRED_2D_FLAT = E2.predict(X_TE2)   # (T_TEST * N2,)
print(f"  Prediction time: {time.time()-t0:.1f}s")

# Reshape to (T_TEST, N)
PRED_1D = PRED_1D_FLAT.reshape(T_TEST, N1)  # (445, 17)
PRED_2D = PRED_2D_FLAT.reshape(T_TEST, N2)  # (445, 3716)

print(f"\n  PRED_1D: shape={PRED_1D.shape}  "
      f"min={PRED_1D.min():.3f}  max={PRED_1D.max():.3f}  "
      f"mean={PRED_1D.mean():.3f}")
print(f"  PRED_2D: shape={PRED_2D.shape}  "
      f"min={PRED_2D.min():.3f}  max={PRED_2D.max():.3f}  "
      f"mean={PRED_2D.mean():.3f}")

del X_TE1, X_TE2, PRED_1D_FLAT, PRED_2D_FLAT
gc.collect()


# ════════════════════════════════════════════════════════════════════════════
# CELL 10 ─ FILL SAMPLE SUBMISSION  (vectorised numpy — ~30 seconds)
# ════════════════════════════════════════════════════════════════════════════
# The submission has: row_id, model_id, event_id, node_type, node_id, water_level
# We use ts_rank = cumcount within (model_id, event_id, node_type, node_id) group
# = the timestep index (0, 1, …, 444) for each prediction row.
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  📝  FILLING SUBMISSION  (vectorised numpy)")
print("═" * 70)
print(f"\n  Submission: {SUB.shape}")
print(f"  event_ids : {sorted(SUB['event_id'].unique())}")
print(f"  model_ids : {sorted(SUB['model_id'].unique())}")

sub = SUB.copy()

# Compute timestep rank within each (model, event, node_type, node_id) group
sub = sub.sort_values('row_id').reset_index(drop=True)
sub['ts_rank'] = (sub
    .groupby(['model_id','event_id','node_type','node_id'])
    .cumcount()
    .astype(np.int32))

ts_per_grp = sub.groupby(['model_id','event_id','node_type','node_id'])['ts_rank'].max().max() + 1
print(f"  Timesteps per group: {ts_per_grp}  (test has {T_TEST})")

# Build fast lookup arrays: node_id → column index
idx1d = {int(x): i for i, x in enumerate(NODES_1D)}
idx2d = {int(x): i for i, x in enumerate(NODES_2D)}

max_nid1 = max(idx1d.keys()) + 2
max_nid2 = max(idx2d.keys()) + 2

nid2col1 = np.zeros(max_nid1, dtype=np.int32)
for nid, col in idx1d.items():
    nid2col1[nid] = col

nid2col2 = np.zeros(max_nid2, dtype=np.int32)
for nid, col in idx2d.items():
    nid2col2[nid] = col

# Numpy arrays for vectorised fill
nt_arr  = sub['node_type'].values.astype(np.int32)
nid_arr = sub['node_id'].values.astype(np.int32)
ts_arr  = np.clip(sub['ts_rank'].values.astype(np.int32), 0, T_TEST - 1)

wl_out = np.empty(len(sub), dtype=np.float32)

print("  Filling type-1 (1D) nodes …")
mask1 = (nt_arr == 1)
if mask1.any():
    nids1 = np.clip(nid_arr[mask1], 0, max_nid1 - 1)
    wl_out[mask1] = PRED_1D[ts_arr[mask1], nid2col1[nids1]]

print("  Filling type-2 (2D) nodes …")
mask2 = (nt_arr == 2)
if mask2.any():
    nids2 = np.clip(nid_arr[mask2], 0, max_nid2 - 1)
    wl_out[mask2] = PRED_2D[ts_arr[mask2], nid2col2[nids2]]

sub['water_level'] = wl_out


# ════════════════════════════════════════════════════════════════════════════
# CELL 11 ─ VALIDATION
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  🔍  SUBMISSION VALIDATION")
print("═" * 70)

required = ['row_id','model_id','event_id','node_type','node_id','water_level']
sub_out  = sub[required].copy()
expected = 50_910_192
actual   = len(sub_out)
wl       = sub_out['water_level']

print(f"\n  {'✅' if actual==expected else '⚠️'}  Rows : {actual:,} / {expected:,}")
print(f"  Cols : {list(sub_out.columns)}")
print(f"\n  Water Level Statistics:")
print(f"    Min  = {wl.min():>12.4f}")
print(f"    Max  = {wl.max():>12.4f}")
print(f"    Mean = {wl.mean():>12.4f}")
print(f"    Std  = {wl.std():>12.4f}")
print(f"    NaN  = {wl.isna().sum():>12,}")
print(f"    Neg  = {(wl < 0).sum():>12,}")
print(f"    Inf  = {np.isinf(wl.values).sum():>12,}")

print(f"\n  By node type:")
for nt in [1, 2]:
    m = sub_out['node_type'] == nt
    print(f"    Type {nt}: {m.sum():>12,} rows  "
          f"mean={sub_out.loc[m,'water_level'].mean():.4f}  "
          f"std={sub_out.loc[m,'water_level'].std():.4f}")

print(f"\n  By model_id:")
for mid in sorted(sub_out['model_id'].unique()):
    m = sub_out['model_id'] == mid
    print(f"    Model {mid}: {m.sum():>12,} rows  "
          f"mean={sub_out.loc[m,'water_level'].mean():.4f}")

print(f"\n  By event_id (sample):")
for eid in sorted(sub_out['event_id'].unique())[:5]:
    m = sub_out['event_id'] == eid
    print(f"    Event {eid}: {m.sum():>10,} rows  "
          f"mean={sub_out.loc[m,'water_level'].mean():.4f}")

# Sanity: check OOF score approximation
oof_score_2d = std_rmse(
    np.tile(PIV_TR2['water_level'].flatten(),1),
    E2.oof[:len(PIV_TR2['water_level'].flatten())],
    E2.sigma
) if E2.oof is not None else float('nan')
print(f"\n  OOF Std-RMSE 2D (approx): {oof_score_2d:.6f}")


# ════════════════════════════════════════════════════════════════════════════
# CELL 12 ─ SAVE FILES
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  💾  SAVING OUTPUTS")
print("═" * 70)

os.makedirs(CFG.OUT, exist_ok=True)
model_dir = f"{CFG.OUT}/models"
os.makedirs(model_dir, exist_ok=True)

# ── Submission ────────────────────────────────────────────────────────────
csv_path = f"{CFG.OUT}/submission.csv"
zip_path = f"{CFG.OUT}/submission.zip"
sub_out.to_csv(csv_path, index=False)
sub_out.to_csv(zip_path, index=False, compression='zip')
csv_mb = os.path.getsize(csv_path) / 1e6
zip_mb = os.path.getsize(zip_path) / 1e6
print(f"  ✅ submission.csv : {csv_mb:.2f} MB")
print(f"  ✅ submission.zip : {zip_mb:.2f} MB")

# ── Models ────────────────────────────────────────────────────────────────
for nt, E in [(1, E1), (2, E2)]:
    for fi, (lm, xm, rm, sc) in enumerate(zip(
            E.lgb_models, E.xgb_models, E.rdg_models, E.rdg_scalers)):
        lm.booster_.save_model(f"{model_dir}/lgb_nt{nt}_f{fi}.txt")
        xm.save_model(f"{model_dir}/xgb_nt{nt}_f{fi}.json")
        joblib.dump(rm, f"{model_dir}/rdg_nt{nt}_f{fi}.pkl")
        joblib.dump(sc, f"{model_dir}/scaler_nt{nt}_f{fi}.pkl")

meta = {
    'feat_names_1d': FN1, 'feat_names_2d': FN2,
    'sigma_1d': E1.sigma, 'sigma_2d': E2.sigma,
    'T_train': T_TRAIN, 'T_test': T_TEST,
    'N1': int(N1), 'N2': int(N2),
    'strategy': 'direct_forcing_no_AR',
    'blend': {'lgb': CFG.W_LGB, 'xgb': CFG.W_XGB, 'rdg': CFG.W_RDG},
}
with open(f"{model_dir}/meta.json", 'w') as f:
    json.dump(meta, f, indent=2)

print(f"  ✅ Models → {model_dir}/")


# ════════════════════════════════════════════════════════════════════════════
# CELL 13 ─ FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  🏆  FINAL SUMMARY")
print("═" * 70)
print(f"""
  ┌────────────────────────────────────────────────────────────────────┐
  │  Strategy : Direct Forcing Prediction (no AR error accumulation)   │
  │  Models   : LGB {CFG.W_LGB:.0%} + XGB {CFG.W_XGB:.0%} + Ridge {CFG.W_RDG:.0%}  ×{CFG.N_FOLDS}-fold     │
  │  Features : {len(FN1)} (1D)  /  {len(FN2)} (2D)                     │
  │             rainfall_cum, peak, lag, static, temporal, spatial      │
  │  1D sigma : {E1.sigma:.4f}                                           │
  │  2D sigma : {E2.sigma:.4f}                                           │
  ├────────────────────────────────────────────────────────────────────┤
  │  Rows     : {actual:>15,}                                  │
  │  CSV      : {csv_mb:>10.2f} MB                                     │
  │  ZIP      : {zip_mb:>10.2f} MB                                     │
  └────────────────────────────────────────────────────────────────────┘

  SUBMIT:
    !kaggle competitions submit \\
        -c urban-flood-modelling \\
        -f /kaggle/working/submission.csv \\
        -m "v6 Direct Forcing LGB+XGB+Ridge — no AR"
""")
print("═" * 70)
print("  ✅  READY — GOOD LUCK!  🏆")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
  🏆  URBANFLOODBENCH — ELITE SOLUTION  v6.0
  Strategy: Direct Forcing Prediction (no AR divergence)
══════════════════════════════════════════════════════════════════════
  Python 3.12.12  |  LGB 4.6.0  |  XGB 3.1.3
══════════════════════════════════════════════════════════════════════

📂  Loading competition data …

  Loading 25 files …


  Reading: 100%|██████████| 25/25 [00:20<00:00,  1.25it/s]


  ✅ 25 files  |  745.1 MB

  Training timesteps : 94
  Test timesteps     : 445
  Submission rows    : 50,910,192
  Sample submission cols: ['row_id', 'model_id', 'event_id', 'node_type', 'node_id', 'water_level']
  model_ids : [np.int8(1), np.int8(2)]
  event_ids : [np.int8(4), np.int8(5), np.int8(8), np.int8(17), np.int8(18), np.int8(22), np.int8(26), np.int8(29), np.int8(31), np.int8(33), np.int8(35), np.int8(37), np.int8(42), np.int8(44), np.int8(48), np.int8(51), np.int8(52), np.int8(53), np.int8(54), np.int8(59), np.int8(60), np.int8(61), np.int8(62), np.int8(65), np.int8(66), np.int8(67), np.int8(69), np.int8(73), np.int8(75), np.int8(76), np.int8(77), np.int8(80), np.int8(81), np.int8(82), np.int8(83), np.int8(84), np.int8(88), np.int8(90), np.int8(97), np.int8(99)]

  1D nodes: 17  |  2D nodes: 3716
  Neighbour matrices: (17, 6)  (3716, 8)
  Static 1D: (17, 6)  Static 2D: (3716, 9)

  Pivoting data to wide format …
  ✅ Pivot done in 1.1s
    TR1: [('water_level', (94, 17)), ('